In [30]:
# ========== 导入：把后面要用的工具箱搬进来 ==========
# 本练习（第 2 周作业）：抓取网页内容 → 用本地 Ollama（OpenAI 兼容接口）做印地语摘要

# 从同目录 scraper 模块导入 fetch_website_contents：抓取并返回网站正文文本
from scraper import fetch_website_contents
# 从 openai 导入 OpenAI 客户端类：既可连云端 OpenAI，也可通过 base_url 连本地 Ollama
from openai import OpenAI

In [31]:
# ========== 客户端：一条云端、一条本地 Ollama（OpenAI-compatible） ==========

# 创建默认 OpenAI 客户端：密钥从环境变量 OPENAI_API_KEY 读取（本格后面主要用 ollama）
openai = OpenAI()
# 本地 Ollama 的 OpenAI 兼容基址：/v1 表示走 Chat Completions 兼容层（不是原生 /api/chat）
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 指向 Ollama 的客户端：base_url 改本地；api_key 填占位即可（本地一般不校验，但 SDK 要求有值）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="anything")

In [32]:
# ========== system_prompt：定角色与输出格式（发给模型的字符串保持英文原样） ==========
# 理念：system 告诉模型「你是谁、怎么答」；这里要求直接输出 Markdown，不要包在代码块里

system_prompt="""
    you are my persional assistent and having vast knowledge and can analyse the contents of any website very quickly and reliably.
    Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [33]:
# ========== user_prompt：任务说明前缀（真正网页正文会在后面拼接） ==========
# 要求：用印地语（hindi）做短摘要；若有新闻/公告也一并概括——prompt 原文不翻译

user_prompt="""
    Here are the contents of a website.
    Provide a short summary of this website in hindi.
    If it includes news or announcements, then summarize these too.
"""

In [34]:
# ========== messages_for：把网页正文装配成 Chat Completions 的 messages 列表 ==========
# 返回格式与 API 要求一致：system 一条 + user 一条（user = 任务说明 + 网站内容）

def messages_for(website):
    # website：抓取到的站点文本；拼进 user.content，供模型阅读后摘要
    return [
        # role=system：注入上面的 system_prompt（角色与格式约束）
        {"role": "system", "content": system_prompt},
        # role=user：user_prompt 前缀 + 网页正文；顺序是「先说明任务，再给材料」
        {"role": "user", "content": user_prompt + website}
    ]

In [ ]:
# ========== 冒烟测试：先不抓网页，直接问本地 llama3.2 一条职业建议 ==========
# 目的：确认 Ollama OpenAI 兼容接口、模型名与 messages 结构都能跑通

# message：手工拼的 messages 列表（注意变量名是 message，不是 messages）
message = [
    {
        # system：简短人设（字符串保持英文原样，不改写）
        "role": "system",
        "content": "you are a smart and friendly assistent."
    },
    {
        # user：具体问题——快速增长的科技/AI 时代求职业建议
        "role": "user",
        "content": "can you give a career advice for this fast growing tech industry and AI era?"
    }
]
# 调用本地 Ollama：model 必须是本机已 pull 的名字；messages 传上面的列表
response = ollama.chat.completions.create(model="llama3.2", messages=message)
# 从 choices[0].message.content 取出助手回复正文并打印
print(response.choices[0].message.content)

In [36]:
# ========== summary(url)：抓站 → 组 messages → 本地模型摘要 ==========
# 这是作业主流程：把 scraper + prompts + Ollama 串成一条函数

def summary(url):
    # 1) 抓取网址正文（依赖 scraper.fetch_website_contents）
    website = fetch_website_contents(url)
    # 2) 用 OpenAI 兼容接口调本地 llama3.2；messages 由 messages_for 生成
    response = ollama.chat.completions.create(
        # 模型 id 保持原样：需本机 ollama pull llama3.2
        model="llama3.2",
        # system + user（含网页内容）
        messages=messages_for(website)
    )
    # 3) 只返回助手文本，方便上层 print / 展示
    return response.choices[0].message.content

In [ ]:
# ========== 实跑：对目标站点调用 summary 并打印印地语摘要 ==========
# URL 保持原样；需本机 Ollama 在跑，且 scraper 能访问该站

print(summary("https://ceratattva.com/"))